# 🌟 Distributed Computing in ELF: A Friendly Guide to JAX Multi-Device Training

Welcome! This notebook is designed to help you understand how **distributed computing** is implemented in the ELF codebase. JAX handles multi-device parallelization very differently from PyTorch (which uses DDP or FSDP). 

Instead of hiding device distribution behind high-level wrappers, JAX exposes explicit primitives like `pmap` (Parallel Map) and collective communications. This guide makes those concepts simple, visual, and interactive.

## 1. Key Concepts in JAX Distributed Computing

Here are the core building blocks you'll find in the ELF codebase (`train.py` and `train_step.py`):

| Concept | What it is | How it's used in ELF |
| :--- | :--- | :--- |
| **Multi-Host Init** | Initializing network coordination between separate machines. | `jax.distributed.initialize()` |
| **Parallel Map (`pmap`)** | Compiles and runs a function across multiple devices in parallel. | `p_train_step = jax.pmap(train_step, axis_name='batch')` |
| **Replication** | Copying variables (like model weights) across all devices. | `state = jax_utils.replicate(state)` |
| **Sharding** | Splitting a data batch so each device gets a fraction of it. | `batch = shard(batch)` |
| **Collective Sync (`pmean`)** | Averaging gradients/losses across all devices (All-Reduce). | `grads = jax.lax.pmean(grads, axis_name='batch')` |
| **Device-Specific RNG** | Ensuring each device generates unique random noise/dropout masks. | `jax.random.fold_in(rng, jax.lax.axis_index('batch'))` |

## 2. Interactive Setup: Simulating Multi-Device JAX

Even if you only have a single CPU or GPU, JAX allows us to **simulate multiple virtual devices** by setting an environment variable before importing JAX. Let's spawn 4 virtual CPU devices to play with!

In [ ]:
import os
# Force JAX/XLA to simulate 4 CPU devices
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"

import jax
import jax.numpy as jnp
from flax import jax_utils
from flax.training.common_utils import shard

print("Number of available devices:", jax.device_count())
print("Devices:", jax.devices())

## 3. Concept 1: Parallel Map (`jax.pmap`)

In JAX, `jax.pmap` compiles a function using XLA and executes it in parallel across your devices.

### How shape changes:
If a standard function takes inputs of shape `(Batch, ...)`, a `pmap`'ed function expects inputs of shape `(Num_Devices, Batch_per_Device, ...)`. The leading dimension is mapped directly to the devices.

In [ ]:
# A simple function that squares its input
def square_fn(x):
    return x ** 2

# Parallel-map the function across our 4 devices
p_square = jax.pmap(square_fn)

# Create an input of shape (4,) - one number for each of our 4 devices
x = jnp.array([1.0, 2.0, 3.0, 4.0])

result = p_square(x)
print("Input:", x)
print("Parallelized Output:", result)
print("Result type:", type(result))  # It resides on the device

## 4. Concept 2: Parameter Replication (`flax.jax_utils.replicate`)

During training, each device needs a copy of the model weights to perform forward and backward passes. 
We initialize weights once, and then use `replicate()` to broadcast them to all devices.

This adds a leading dimension of size `num_devices` to every weight tensor.

In [ ]:
# A mock model parameter dictionary
params = {
    "kernel": jnp.array([[1.0, 2.0], [3.0, 4.0]]),
    "bias": jnp.array([0.5, -0.5])
}
print("Original bias shape:", params["bias"].shape)

# Replicate the parameters to 4 devices
replicated_params = jax_utils.replicate(params)
print("Replicated bias shape:", replicated_params["bias"].shape)  # shape is now (4, 2)
print("Replicated bias values:", replicated_params["bias"])

## 5. Concept 3: Data Sharding (`flax.training.common_utils.shard`)

To train in parallel, we divide our batch of training data. 
If the global batch size is 8 and we have 4 devices, each device processes 2 samples. 
`shard()` takes a batch dictionary of shape `(Batch_Size, ...)` and reshapes it to `(Num_Devices, Batch_Size // Num_Devices, ...)`. This automatically sends the correct slice to each device.

In [ ]:
# A batch of training data: 8 samples, sequence length 5
batch = {
    "input_ids": jnp.arange(40).reshape(8, 5),
    "attention_mask": jnp.ones((8, 5))
}
print("Original Batch input_ids shape:", batch["input_ids"].shape)

# Shard the batch
sharded_batch = shard(batch)
print("Sharded Batch input_ids shape:", sharded_batch["input_ids"].shape)  # shape is now (4, 2, 5)
print("Device 0's portion:\n", sharded_batch["input_ids"][0])
print("Device 1's portion:\n", sharded_batch["input_ids"][1])

## 6. Concept 4: Collective Communication (`jax.lax.pmean`)

Each device calculates gradients on its own data shard. Because each device sees different data, they get different gradients. 
To keep the weights synchronized, we must average the gradients across all devices. 
In JAX, we do this using `jax.lax.pmean` inside our pmapped training step.

In [ ]:
def train_step_simulation(local_val):
    # Simulate calculating a loss gradient locally on each device
    # Device 0: 2, Device 1: 4, Device 2: 6, Device 3: 8
    local_gradient = local_val * 2

    # All-Reduce average across devices (axis_name matches the pmap axis_name)
    synced_gradient = jax.lax.pmean(local_gradient, axis_name="batch")
    return synced_gradient

p_step = jax.pmap(train_step_simulation, axis_name="batch")

# Pass different inputs to each device
device_inputs = jnp.array([1.0, 2.0, 3.0, 4.0])
# Expected local gradients: [2.0, 4.0, 6.0, 8.0]
# Average gradient: (2 + 4 + 6 + 8) / 4 = 5.0

synced_grads = p_step(device_inputs)
print("Device local inputs:", device_inputs)
print("Synchronized gradients on each device:", synced_grads)

## 7. Concept 5: Device-Unique Randomness

When adding noise (as in diffusion models) or applying dropout, we want **different** random numbers on each device. 
If we pass the same RNG key to all devices, they will do the exact same things, rendering multi-device training redundant.

We fix this by: 
1. Using `jax.lax.axis_index(axis_name)` to get the device's index (0, 1, 2, or 3).
2. Using `jax.random.fold_in(rng, axis_idx)` to derive a unique sub-key for each device.

In [ ]:
def random_noise_step(replicated_rng):
    # 1. Get current device index
    device_idx = jax.lax.axis_index("batch")
    
    # 2. Fold the index into the RNG to get a unique key per device
    unique_device_rng = jax.random.fold_in(replicated_rng, device_idx)
    
    # 3. Generate noise
    return jax.random.normal(unique_device_rng, shape=(2,))

p_random = jax.pmap(random_noise_step, axis_name="batch")

# Replicate a base key across devices
base_key = jax.random.PRNGKey(42)
replicated_keys = jax_utils.replicate(base_key)

noises = p_random(replicated_keys)
print("Noise generated on Device 0:\n", noises[0])
print("Noise generated on Device 1:\n", noises[1])
print("Notice that they are completely different!")

## 8. Putting It All Together: A Complete Toy Training Loop

Let's write a simple linear model training script in parallel. 
This mirrors exactly what the ELF codebase does in `src/train.py` and `src/train_step.py`, but condensed into 30 lines of code.

In [ ]:
import optax
from flax.training.train_state import TrainState

# 1. Define a toy linear model apply function
def model_apply(params, x):
    return x @ params["w"] + params["b"]

# 2. Define the training step (which will be pmapped)
def train_step(state, batch):
    x, y = batch["x"], batch["y"]
    
    # Loss function
    def loss_fn(params):
        preds = model_apply(params, x)
        loss = jnp.mean((preds - y) ** 2)
        return loss
    
    # Compute loss and gradients
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    
    # --- DISTRIBUTED SYNC ---
    # Average grads and loss across all 4 devices
    grads = jax.lax.pmean(grads, axis_name="batch")
    loss = jax.lax.pmean(loss, axis_name="batch")
    # -------------------------
    
    # Update local parameters
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss

# Pmap the train step
p_train_step = jax.pmap(train_step, axis_name="batch")

In [ ]:
# 3. Initialization
init_params = {
    "w": jnp.array([[0.1], [0.2]]),
    "b": jnp.array([0.0])
}
tx = optax.adam(learning_rate=0.1)
state = TrainState.create(apply_fn=model_apply, params=init_params, tx=tx)

# Replicate the TrainState to all 4 devices
replicated_state = jax_utils.replicate(state)

# 4. Generate some mock data (Global Batch Size = 16)
key = jax.random.PRNGKey(0)
x_data = jax.random.normal(key, (16, 2))
true_w = jnp.array([[2.5], [-1.5]])
true_b = jnp.array([0.8])
y_data = x_data @ true_w + true_b + 0.1 * jax.random.normal(key, (16, 1))

global_batch = {"x": x_data, "y": y_data}

# Shard the batch so each device gets 4 samples
sharded_batch = shard(global_batch)
print("Sharded X shape:", sharded_batch["x"].shape)  # (4 devices, 4 samples/device, 2 features)

# 5. Run a few training steps
for step in range(5):
    replicated_state, loss = p_train_step(replicated_state, sharded_batch)
    # The loss is replicated across devices, so we read it from device 0
    print(f"Step {step + 1} - Loss: {loss[0]:.4f}")

# View the final weights on device 0
final_params = jax_utils.unreplicate(replicated_state).params
print("\nTrue Weights (w):", true_w.flatten())
print("Trained Weights (w):", final_params["w"].flatten())
print("True Bias (b):", true_b)
print("Trained Bias (b):", final_params["b"])

## Summary

To recap, here is how the data-parallel execution works in ELF:
1. The training dataset is loaded on the host, and batches are retrieved.
2. `shard(batch)` is called to split the data batch evenly across devices.
3. The weights/train state are replicated to all devices using `replicate(state)` before starting the loop.
4. `p_train_step` is executed. Each device calculates local losses and gradients on its slice of data.
5. Gradients and losses are synchronized and averaged across all devices via `pmean`.
6. Weights are updated locally on each device. Because they all applied the same averaged gradient, the weights on each device remain perfectly in sync!